In [ ]:
# scripts/05_train_baseline.py

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import os

# Paths
DATA_FILE = "data/processed/mutation_features.csv"
os.makedirs("data/models", exist_ok=True)

# Load dataset
df = pd.read_csv(DATA_FILE)

# Features
embedding_cols = [str(i) for i in range(1280)]  # ESM embedding columns
optional_dms_cols = ["bind_lib1", "bind_lib2", "ace2_score", "expr_lib1", "expr_lib2", "expr_score"]
feature_cols = embedding_cols + ["embedding_missing"] + optional_dms_cols

# Some columns might not exist
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y = df["label"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize Random Forest
clf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15, 
    random_state=42,
    n_jobs=-1
)

# Train
print("Training Random Forest classifier...")
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["deleterious", "neutral"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Optional: save trained model
import joblib
MODEL_FILE = "data/models/rf_baseline.pkl"
joblib.dump(clf, MODEL_FILE)
print(f"\nModel saved to {MODEL_FILE}")


In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# =======================
# Load dataset
# =======================
DATA_DIR = "data/processed"
features_file = os.path.join(DATA_DIR, "mutation_features.csv")

print("Loading dataset...")
df = pd.read_csv(features_file)

print(f"Shape: {df.shape}")
print(df["label"].value_counts())

# =======================
# Split features / labels
# =======================
X = df.drop(columns=["mut_id", "label"])
y = df["label"]

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

print("Final feature set shape:", X.shape)
print("Example dtypes:\n", X.dtypes.head())

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# =======================
# Train Random Forest
# =======================
print("\nTraining Random Forest classifier...")
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# =======================
# Evaluation
# =======================
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

# =======================
# Cross-validation
# =======================
print("\nRunning 5-fold cross-validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
print("CV scores:", cv_scores)
print("Mean CV accuracy:", np.mean(cv_scores))

# =======================
# Feature importance
# =======================
print("\nTop 20 feature importances:")
importances = pd.Series(clf.feature_importances_, index=X.columns)
top_feats = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(8, 6))
sns.barplot(x=top_feats.values, y=top_feats.index, palette="viridis")
plt.title("Top 20 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

# =======================
# Embedding visualization (PCA)
# =======================
print("\nRunning PCA for visualization...")
embedding_cols = [c for c in X.columns if c.isdigit()]  # only ESM embeddings
X_embed = X[embedding_cols].fillna(0.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_embed)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=y, alpha=0.7, palette={"neutral": "blue", "deleterious": "red"}
)
plt.title("PCA of Mutation Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Label")
plt.show()


In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from cuml.ensemble import RandomForestClassifier as cuRF
from cuml.model_selection import train_test_split  # GPU accelerated
from sklearn.model_selection import StratifiedKFold  # for CV
from cuml.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance

# =======================
# Load dataset
# =======================
features_file = "/mutation_features.csv"
print("Loading dataset...")
df = pd.read_csv(features_file)

print(f"Shape: {df.shape}")
print(df["label"].value_counts())

# =======================
# Split features / labels
# =======================
X = df.drop(columns=["mut_id", "label"])
y = df["label"]

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

print("Final feature set shape:", X.shape)
print("Example dtypes:\n", X.dtypes.head())

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# =======================
# Train Random Forest (GPU)
# =======================
print("\nTraining GPU Random Forest classifier...")
clf = cuRF(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_bins=128,
    bootstrap=True
)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# =======================
# Evaluation
# =======================
print("\nClassification Report:")
print(classification_report(y_test.to_numpy(), y_pred.to_numpy()))

cm = confusion_matrix(y_test.to_numpy(), y_pred.to_numpy(), labels=np.unique(y))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y))
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

# =======================
# Cross-validation (manual loop with cuML)
# =======================
print("\nRunning 5-fold cross-validation (manual)...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []
for train_idx, val_idx in cv.split(X, y):
    X_train_cv, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
    y_train_cv, y_val_cv = y.iloc[train_idx], y.iloc[val_idx]

    model_cv = cuRF(
        n_estimators=300,
        max_depth=16,
        random_state=42,
        n_bins=128,
        bootstrap=True
    )
    model_cv.fit(X_train_cv, y_train_cv)
    y_val_pred = model_cv.predict(X_val_cv)
    acc = accuracy_score(y_val_cv, y_val_pred)
    cv_scores.append(acc)

print("CV scores:", cv_scores)
print("Mean CV accuracy:", np.mean(cv_scores))

# =======================
# Feature importance (Permutation importance)
# =======================
print("\nTop 20 feature importances:")
print("cuML RF has no native feature_importances_, using permutation importance (CPU)...")

result = permutation_importance(
    clf, X_test, y_test,
    n_repeats=5, random_state=42, n_jobs=-1
)
importances = pd.Series(result.importances_mean, index=X.columns)

top_feats = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(8, 6))
sns.barplot(
    x=top_feats.values,
    y=top_feats.index,
    hue=top_feats.index,
    dodge=False,
    legend=False,
    palette="viridis"
)
plt.title("Top 20 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

# =======================
# Embedding visualization (PCA)
# =======================
print("\nRunning PCA for visualization...")
embedding_cols = [c for c in X.columns if c.isdigit()]  # only embedding cols
X_embed = X[embedding_cols].fillna(0.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_embed)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Map labels for nicer legend
label_map = {0: "neutral", 1: "deleterious"}
y_mapped = y.map(label_map)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=y_mapped, alpha=0.7,
    palette={"neutral": "blue", "deleterious": "red"}
)
plt.title("PCA of Mutation Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Label")
plt.show()
